In [1]:
from transformers import pipeline

# Load emotion classifier
classifier = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

e:\7. Projects From Sem 3\TTS\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 249.75it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def init_engine():
    engine = pyttsx3.init('sapi5')
    voices = engine.getProperty("voices")
    soft_voice_id = voices[1].id if len(voices) > 1 else voices[0].id
    default_voice_id = voices[0].id
    return engine, soft_voice_id, default_voice_id

# CUSTOM EMOTION VOICES (anger now FASTER)
def apply_love_voice(engine, soft_voice_id):
    """Soft, soothing 'love' voice (unchanged - perfect)"""
    base_rate = engine.getProperty("rate")
    love_rate = max(130, min(170, base_rate - 30))  # slow 140-160 wpm
    
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", love_rate)
    engine.setProperty("volume", 0.9)
    print(f"[LOVE] rate={love_rate}, volume=0.9")

def apply_anger_voice(engine, default_voice_id):
    """Firm, FAST 'anger' voice (sped up to 210-240 wpm)"""
    base_rate = engine.getProperty("rate")
    anger_rate = max(210, min(240, base_rate + 50))  # FASTER: +50 instead of +20
    
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", anger_rate)
    engine.setProperty("volume", 1.0)
    print(f"[ANGER] FAST rate={anger_rate}, volume=1.0")

def classify_emotion(text: str):
    scores = classifier(text)[0]
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    top = scores[0]
    print("\nAll emotions:")
    for r in scores:
        print(f"{r['label']:10s} -> {r['score']:.3f}")
    print(f"Top emotion: {top['label']} ({top['score']:.2f})")
    return top["label"], top["score"]

def speak_with_emotion(text: str, filename: str = "output.wav"):
    print(f"\nInput text: {text}")
    emotion, confidence = classify_emotion(text)

    engine, soft_voice_id, default_voice_id = init_engine()
    print("✓ Engine initialized")

    # Route to custom voices
    if emotion == "joy" and confidence > 0.7:
        apply_love_voice(engine, soft_voice_id)
    elif emotion == "anger":
        apply_anger_voice(engine, default_voice_id)  # Now FASTER
    else:
        # neutral fallback
        base_rate = engine.getProperty("rate")
        neutral_rate = max(150, min(190, base_rate))
        engine.setProperty("rate", neutral_rate)
        engine.setProperty("volume", 0.9)
        print(f"[NEUTRAL] rate={neutral_rate}")

    # Generate audio
    engine.save_to_file(text, filename)
    engine.runAndWait()
    print(f"✅ Audio saved: {filename}")

# Test both
if __name__ == "__main__":
    speak_with_emotion("I love you sweetie, you mean everything to me.", "love_soft.wav")
    speak_with_emotion("You can go to hell, what do you think you are doing?!", "anger_fast.wav")


Input text: I love you sweetie, you mean everything to me.

All emotions:
joy        -> 0.928
neutral    -> 0.032
anger      -> 0.015
surprise   -> 0.009
sadness    -> 0.009
disgust    -> 0.006
fear       -> 0.001
Top emotion: joy (0.93)
✓ Engine initialized
[LOVE] rate=170, volume=0.9
✅ Audio saved: love_soft.wav

Input text: You can go to hell, what do you think you are doing?!

All emotions:
anger      -> 0.799
disgust    -> 0.144
surprise   -> 0.020
fear       -> 0.014
neutral    -> 0.010
sadness    -> 0.010
joy        -> 0.003
Top emotion: anger (0.80)
✓ Engine initialized
[ANGER] FAST rate=220, volume=1.0
✅ Audio saved: anger_fast.wav


In [1]:
from transformers import pipeline
import pyttsx3
import time
import os

# Load emotion classifier (detects: joy, anger, sadness, fear, surprise, disgust, neutral)
classifier = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

def init_engine():
    """Initialize TTS engine with voice selection"""
    engine = pyttsx3.init('sapi5')
    voices = engine.getProperty("voices")
    soft_voice_id = voices[1].id if len(voices) > 1 else voices[0].id  # Female-ish for love
    default_voice_id = voices[0].id  # Default for anger/other
    return engine, soft_voice_id, default_voice_id

def classify_emotion(text):
    """Detect emotion from text with detailed output"""
    scores = sorted(classifier(text)[0], key=lambda x: x["score"], reverse=True)
    top = scores[0]
    
    print(f"\n🎭 TESTING: '{text}'")
    print("All emotions:")
    for r in scores:
        print(f"  {r['label']:10s} -> {r['score']:.3f}")
    print(f"📊 TOP: {top['label']} ({top['score']:.2f})")
    print("-" * 60)
    
    return top["label"], top["score"]

def speak_joy(engine, soft_voice_id, text, filename):
    """JOY: Bright, energetic, happy (160-190 wpm + lively pauses)"""
    base_rate = engine.getProperty("rate")
    joy_rate = max(160, min(190, base_rate + 10))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", joy_rate)
    engine.setProperty("volume", 0.95)
    
    print("🌟 JOY SETTINGS: rate={} wpm, vol=0.95".format(joy_rate))
    save_emotional_speech(engine, text, filename)

def speak_anger(engine, default_voice_id, text, filename):
    """ANGER: Fast, firm, intense (210-240 wpm)"""
    base_rate = engine.getProperty("rate")
    anger_rate = max(210, min(240, base_rate + 50))
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", anger_rate)
    engine.setProperty("volume", 1.0)
    
    print("🔥 ANGER SETTINGS: rate={} wpm, vol=1.0".format(anger_rate))
    save_emotional_speech(engine, text, filename)

def speak_sadness(engine, soft_voice_id, text, filename):
    """SADNESS: Slow, soft, melancholic (120-150 wpm)"""
    base_rate = engine.getProperty("rate")
    sad_rate = max(120, min(150, base_rate - 50))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", sad_rate)
    engine.setProperty("volume", 0.6)
    
    print("😢 SADNESS SETTINGS: rate={} wpm, vol=0.6".format(sad_rate))
    save_emotional_speech(engine, text, filename)

def speak_fear(engine, soft_voice_id, text, filename):
    """FEAR: Hesitant, shaky, urgent (140-170 wpm)"""
    base_rate = engine.getProperty("rate")
    fear_rate = max(140, min(170, base_rate - 20))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", fear_rate)
    engine.setProperty("volume", 0.7)
    
    print("😱 FEAR SETTINGS: rate={} wpm, vol=0.7".format(fear_rate))
    save_emotional_speech(engine, text, filename)

def speak_surprise(engine, soft_voice_id, text, filename):
    """SURPRISE: Fast, high-energy, excited (200-230 wpm)"""
    base_rate = engine.getProperty("rate")
    surprise_rate = max(200, min(230, base_rate + 40))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", surprise_rate)
    engine.setProperty("volume", 1.0)
    
    print("😲 SURPRISE SETTINGS: rate={} wpm, vol=1.0".format(surprise_rate))
    save_emotional_speech(engine, text, filename)

def speak_disgust(engine, default_voice_id, text, filename):
    """DISGUST: Slow, disdainful, nasal (130-160 wpm)"""
    base_rate = engine.getProperty("rate")
    disgust_rate = max(130, min(160, base_rate - 25))
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", disgust_rate)
    engine.setProperty("volume", 0.8)
    
    print("🤢 DISGUST SETTINGS: rate={} wpm, vol=0.8".format(disgust_rate))
    save_emotional_speech(engine, text, filename)

def speak_neutral(engine, default_voice_id, text, filename):
    """NEUTRAL: Normal conversational (170 wpm)"""
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", 170)
    engine.setProperty("volume", 0.85)
    
    print("➖ NEUTRAL SETTINGS: rate=170 wpm, vol=0.85")
    save_emotional_speech(engine, text, filename)

def save_emotional_speech(engine, text, filename):
    """Save emotional speech to WAV file (NO PLAYBACK)"""
    print(f"Saving to {filename}...")
    engine.save_to_file(text, filename)
    engine.runAndWait()
    print(f"✅ SAVED: {filename}")

def speak_with_emotion(text, filename="output.wav"):
    """Main function: emotion → custom voice → SAVE FILE"""
    emotion, confidence = classify_emotion(text)
    
    engine, soft_voice_id, default_voice_id = init_engine()
    
    # Route to custom emotion logic
    emotion_handlers = {
        "joy": lambda: speak_joy(engine, soft_voice_id, text, filename),
        "anger": lambda: speak_anger(engine, default_voice_id, text, filename),
        "sadness": lambda: speak_sadness(engine, soft_voice_id, text, filename),
        "fear": lambda: speak_fear(engine, soft_voice_id, text, filename),
        "surprise": lambda: speak_surprise(engine, soft_voice_id, text, filename),
        "disgust": lambda: speak_disgust(engine, default_voice_id, text, filename),
        "neutral": lambda: speak_neutral(engine, default_voice_id, text, filename)
    }
    
    handler = emotion_handlers.get(emotion, emotion_handlers["neutral"])
    handler()
    print(f"\n🎵 FILE READY: {filename}")

# 🔥 COMPREHENSIVE EMOTION TESTING (SAVES FILES ONLY)
if __name__ == "__main__":
    print("🚀 EMPATHY ENGINE - FULL EMOTION TEST (FILES ONLY)")
    print("=" * 70)
    
    test_cases = [
        ("I love you sweetie, you mean everything to me!", "joy_test.wav"),
        ("You can go to HELL! What are you doing?!", "anger_test.wav"),
        ("I'm so sorry... I feel terrible about this.", "sadness_test.wav"),
        ("Oh no! There's a spider! Help me quick!", "fear_test.wav"),
        ("Wow! I can't believe you did that! Amazing!", "surprise_test.wav"),
        ("That's disgusting! Get that away from me!", "disgust_test.wav"),
        ("The weather is nice today. How are you?", "neutral_test.wav"),
    ]
    
    for text, filename in test_cases:
        speak_with_emotion(text, filename)
        print("\n" + "="*70 + "\n")
    
    print("🎉 ALL 7 EMOTION FILES SAVED!")
    print("📁 Check your folder for: joy_test.wav, anger_test.wav, etc.")
    print("\n🎧 PLAY FILES IN ANY AUDIO PLAYER TO TEST")
    print("💡 Tell me which ones need tuning (rate/volume)!")


e:\7. Projects From Sem 3\TTS\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 225.60it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 EMPATHY ENGINE - FULL EMOTION TEST (FILES ONLY)

🎭 TESTING: 'I love you sweetie, you mean everything to me!'
All emotions:
  joy        -> 0.930
  anger      -> 0.024
  surprise   -> 0.018
  neutral    -> 0.017
  sadness    -> 0.006
  disgust    -> 0.004
  fear       -> 0.001
📊 TOP: joy (0.93)
------------------------------------------------------------
🌟 JOY SETTINGS: rate=190 wpm, vol=0.95
Saving to joy_test.wav...
✅ SAVED: joy_test.wav

🎵 FILE READY: joy_test.wav



🎭 TESTING: 'You can go to HELL! What are you doing?!'
All emotions:
  anger      -> 0.503
  surprise   -> 0.311
  disgust    -> 0.138
  neutral    -> 0.016
  sadness    -> 0.012
  fear       -> 0.011
  joy        -> 0.009
📊 TOP: anger (0.50)
------------------------------------------------------------
🔥 ANGER SETTINGS: rate=240 wpm, vol=1.0
Saving to anger_test.wav...
✅ SAVED: anger_test.wav

🎵 FILE READY: anger_test.wav



🎭 TESTING: 'I'm so sorry... I feel terrible about this.'
All emotions:
  sadness    -> 0.877
  f